# Experiment: Inspect MultiLepPAT GEN-RECO Matching And Decay Chains

This notebook inspects a `MultiLepPAT` ntuple with emphasis on:

1. event-level GEN truth retention,
2. reco-to-GEN muon matching provenance, and
3. index-based GEN decay-chain tracking using `MC_GenPart_motherGenIdx`.

The working convention is that the ntuple row index is the `gen_idx`. The new `MC_GenPart_motherGenIdx` branch allows the mother chain to be walked directly inside the stored flat GEN list.


## Notebook Outline

- Open `mkcands/X_data` from the requested ntuple or a validation fallback.
- Verify the new GEN branch inventory, especially `MC_GenPart_motherGenIdx`.
- Summarize GEN retention and reco muon GEN matching.
- Inspect one event in detail, including parent-child GEN links.
- Reconstruct decay chains and map muons to J/psi or Upsilon, and kaons to phi, by walking `motherGenIdx`.

If the primary ntuple does not yet contain `MC_GenPart_motherGenIdx`, rerun the analyzer after the GEN-chain update before using this notebook on that file.


In [ ]:
from collections import Counter, defaultdict
from math import hypot, isfinite
from pathlib import Path

import ROOT
from IPython.display import display

try:
    import pandas as pd
except ImportError:
    pd = None

ROOT.gROOT.SetBatch(True)

PRIMARY_NTUPLE = Path("/eos/user/c/chiw/JpsiJpsiPhi/CMSSW_15_0_15_JpsiJpsiPhi_refactor/src/mymultilep_JpsiUpsPhi_mcRun2022_TPS.root")
VALIDATION_NTUPLE = Path("/tmp/mother_genidx_validation_numEvent5.root")
TREE_PATH = "mkcands/X_data"

NTUPLE_PATH = PRIMARY_NTUPLE if PRIMARY_NTUPLE.exists() else VALIDATION_NTUPLE
if not NTUPLE_PATH.exists():
    raise FileNotFoundError("Set NTUPLE_PATH to a valid MultiLepPAT ntuple before running the notebook.")

root_file = ROOT.TFile.Open(str(NTUPLE_PATH), "READ")
tree = root_file.Get(TREE_PATH)
if not tree:
    raise RuntimeError(f"Could not find {TREE_PATH} in {NTUPLE_PATH}")

branches = {branch.GetName() for branch in tree.GetListOfBranches()}
required = {"MC_GenPart_pdgId", "MC_GenPart_motherGenIdx", "MC_GenPart_handleIndex"}
missing = sorted(required - branches)
if missing:
    raise RuntimeError(f"Missing required branches: {missing}")

print(f"Using ntuple: {NTUPLE_PATH}")
print(f"Entries in {TREE_PATH}: {tree.GetEntries()}")


In [ ]:
for name in sorted(branch for branch in branches if branch.startswith("MC_GenPart") or branch.startswith("muGenMatch")):
    print(name)


In [ ]:
PDG_LABELS = {
    443: "J/psi", -443: "J/psi",
    553: "Upsilon", -553: "Upsilon",
    333: "phi", -333: "phi",
    13: "mu-", -13: "mu+",
    321: "K+", -321: "K-",
}
MATCH_SOURCE_LABELS = {0: "unmatched", 1: "patRef", 2: "chi2Fallback"}

def pdg_label(pdg_id):
    return PDG_LABELS.get(int(pdg_id), str(int(pdg_id)))

def as_list(vec):
    return list(vec) if vec is not None else []

def maybe_frame(rows):
    if pd is not None:
        return pd.DataFrame(rows)
    return rows

def walk_mother_chain(gen_idx, mother_idx):
    chain = []
    seen = set()
    idx = gen_idx
    while idx >= 0 and idx not in seen:
        chain.append(idx)
        seen.add(idx)
        idx = int(mother_idx[idx])
    return chain

def find_ancestor_of_type(gen_idx, target_abs_pdg_id, pdg_id, mother_idx):
    if gen_idx < 0:
        return -1
    for idx in walk_mother_chain(gen_idx, mother_idx):
        if abs(int(pdg_id[idx])) == target_abs_pdg_id:
            return idx
    return -1

def chain_labels(chain, pdg_id):
    return " -> ".join(f"{idx}:{pdg_label(pdg_id[idx])}" for idx in chain)

def event_snapshot(tree, entry):
    tree.GetEntry(entry)

    gen_pdg_id = as_list(tree.MC_GenPart_pdgId)
    gen_status = as_list(tree.MC_GenPart_status)
    gen_mother_pdg_id = as_list(tree.MC_GenPart_motherPdgId)
    gen_mother_gen_idx = as_list(tree.MC_GenPart_motherGenIdx)
    gen_handle_index = as_list(tree.MC_GenPart_handleIndex)
    gen_pt = as_list(tree.MC_GenPart_pt)
    gen_eta = as_list(tree.MC_GenPart_eta)
    gen_phi = as_list(tree.MC_GenPart_phi)
    gen_mass = as_list(tree.MC_GenPart_mass)

    gen_rows = []
    for idx, pdg_id in enumerate(gen_pdg_id):
        chain = walk_mother_chain(idx, gen_mother_gen_idx)
        gen_rows.append({
            "gen_idx": idx,
            "handle_index": int(gen_handle_index[idx]),
            "pdg_id": int(pdg_id),
            "particle": pdg_label(pdg_id),
            "status": int(gen_status[idx]),
            "mother_pdg_id": int(gen_mother_pdg_id[idx]),
            "mother_gen_idx": int(gen_mother_gen_idx[idx]),
            "mother_particle": pdg_label(gen_mother_pdg_id[idx]),
            "pt": float(gen_pt[idx]),
            "eta": float(gen_eta[idx]),
            "phi": float(gen_phi[idx]),
            "mass": float(gen_mass[idx]),
            "chain": chain_labels(chain, gen_pdg_id),
        })

    mu_rows = []
    mu_px = as_list(tree.muPx)
    mu_py = as_list(tree.muPy)
    mu_pz = as_list(tree.muPz)
    mu_charge = as_list(tree.muCharge)
    mu_gen_idx = as_list(tree.muGenMatchIdx) if hasattr(tree, "muGenMatchIdx") else [-1] * len(mu_px)
    mu_gen_source = as_list(tree.muGenMatchSource) if hasattr(tree, "muGenMatchSource") else [0] * len(mu_px)
    mu_gen_chi2 = as_list(tree.muGenMatchChi2) if hasattr(tree, "muGenMatchChi2") else [-1.0] * len(mu_px)

    for idx, px in enumerate(mu_px):
        match_idx = int(mu_gen_idx[idx]) if idx < len(mu_gen_idx) else -1
        matched_gen = gen_rows[match_idx] if 0 <= match_idx < len(gen_rows) else None
        jpsi_idx = find_ancestor_of_type(match_idx, 443, gen_pdg_id, gen_mother_gen_idx)
        ups_idx = find_ancestor_of_type(match_idx, 553, gen_pdg_id, gen_mother_gen_idx)
        mu_rows.append({
            "mu_idx": idx,
            "charge": int(mu_charge[idx]),
            "pt": hypot(float(px), float(mu_py[idx])),
            "pz": float(mu_pz[idx]),
            "match_idx": match_idx,
            "match_source": MATCH_SOURCE_LABELS.get(int(mu_gen_source[idx]), int(mu_gen_source[idx])),
            "match_chi2": float(mu_gen_chi2[idx]) if idx < len(mu_gen_chi2) and isfinite(float(mu_gen_chi2[idx])) else None,
            "matched_particle": matched_gen["particle"] if matched_gen else None,
            "matched_mother_gen_idx": matched_gen["mother_gen_idx"] if matched_gen else None,
            "matched_chain": matched_gen["chain"] if matched_gen else None,
            "jpsi_ancestor_gen_idx": jpsi_idx,
            "ups_ancestor_gen_idx": ups_idx,
        })

    return {
        "entry": entry,
        "run": int(tree.runNum),
        "lumi": int(tree.lumiNum),
        "event": int(tree.evtNum),
        "nMu": int(tree.nMu),
        "gen_rows": gen_rows,
        "mu_rows": mu_rows,
    }


In [ ]:
summary = {
    "entries": int(tree.GetEntries()),
    "events_gen_nonempty": 0,
    "events_with_parent_links": 0,
    "events_with_muons": 0,
    "events_with_patref_match": 0,
    "events_with_chi2_fallback": 0,
    "total_muons": 0,
    "total_matched_muons": 0,
    "total_parent_links": 0,
}

for entry in range(tree.GetEntries()):
    snap = event_snapshot(tree, entry)
    gen_rows = snap["gen_rows"]
    mu_rows = snap["mu_rows"]
    if gen_rows:
        summary["events_gen_nonempty"] += 1
    if any(row["mother_gen_idx"] >= 0 for row in gen_rows):
        summary["events_with_parent_links"] += 1
    if mu_rows:
        summary["events_with_muons"] += 1
    if any(row["match_source"] == "patRef" for row in mu_rows):
        summary["events_with_patref_match"] += 1
    if any(row["match_source"] == "chi2Fallback" for row in mu_rows):
        summary["events_with_chi2_fallback"] += 1
    summary["total_muons"] += len(mu_rows)
    summary["total_matched_muons"] += sum(row["match_idx"] >= 0 for row in mu_rows)
    summary["total_parent_links"] += sum(row["mother_gen_idx"] >= 0 for row in gen_rows)

maybe_frame([summary])


## Event Browser

Choose an entry and inspect both the flat GEN list and the reco muon matches. The `chain` column is built by following `MC_GenPart_motherGenIdx` upward until the chain terminates.


In [ ]:
ENTRY = 0
snap = event_snapshot(tree, ENTRY)
print({key: snap[key] for key in ["entry", "run", "lumi", "event", "nMu"]})
display(maybe_frame(snap["mu_rows"]))
display(maybe_frame(snap["gen_rows"]))


In [ ]:
def parent_child_decay_rows(tree, entry):
    snap = event_snapshot(tree, entry)
    by_idx = {row["gen_idx"]: row for row in snap["gen_rows"]}
    children = defaultdict(list)

    for row in snap["gen_rows"]:
        parent_idx = row["mother_gen_idx"]
        if parent_idx >= 0:
            children[parent_idx].append(row["gen_idx"])

    rows = []
    for parent_idx in sorted(children):
        parent = by_idx[parent_idx]
        child_indices = children[parent_idx]
        rows.append({
            "parent_gen_idx": parent_idx,
            "parent": parent["particle"],
            "child_gen_indices": child_indices,
            "children": [by_idx[idx]["particle"] for idx in child_indices],
        })
    return rows

display(maybe_frame(parent_child_decay_rows(tree, ENTRY)))


In [ ]:
def matched_muon_decay_rows(tree, entry):
    snap = event_snapshot(tree, entry)
    gen_rows = {row["gen_idx"]: row for row in snap["gen_rows"]}
    rows = []
    for row in snap["mu_rows"]:
        match_idx = row["match_idx"]
        if match_idx < 0:
            continue
        rows.append({
            "mu_idx": row["mu_idx"],
            "match_source": row["match_source"],
            "gen_mu_idx": match_idx,
            "gen_mu_chain": gen_rows[match_idx]["chain"],
            "jpsi_ancestor_gen_idx": row["jpsi_ancestor_gen_idx"],
            "ups_ancestor_gen_idx": row["ups_ancestor_gen_idx"],
        })
    return rows

display(maybe_frame(matched_muon_decay_rows(tree, ENTRY)))


In [ ]:
def resonance_children(tree, entry, resonance_abs_pdg_id):
    snap = event_snapshot(tree, entry)
    gen_rows = snap["gen_rows"]
    pdg_id = [row["pdg_id"] for row in gen_rows]
    mother_idx = [row["mother_gen_idx"] for row in gen_rows]
    rows = []

    for row in gen_rows:
        if abs(row["pdg_id"]) != resonance_abs_pdg_id:
            continue
        daughters = []
        for child in gen_rows:
            chain = walk_mother_chain(child["gen_idx"], mother_idx)
            if row["gen_idx"] in chain[1:]:
                daughters.append(child["gen_idx"])
        rows.append({
            "resonance_gen_idx": row["gen_idx"],
            "resonance": row["particle"],
            "chain": row["chain"],
            "descendant_gen_indices": daughters,
            "descendants": [pdg_label(pdg_id[idx]) for idx in daughters],
        })
    return rows

print("J/psi descendants")
display(maybe_frame(resonance_children(tree, ENTRY, 443)))
print("phi descendants")
display(maybe_frame(resonance_children(tree, ENTRY, 333)))


In [ ]:
def scan_decay_signatures(tree, max_events=None):
    counts = Counter()
    n_entries = tree.GetEntries() if max_events is None else min(tree.GetEntries(), max_events)

    for entry in range(n_entries):
        snap = event_snapshot(tree, entry)
        by_idx = {row["gen_idx"]: row for row in snap["gen_rows"]}
        for row in snap["gen_rows"]:
            parent_idx = row["mother_gen_idx"]
            if parent_idx < 0:
                continue
            parent = by_idx[parent_idx]
            counts[(parent["pdg_id"], row["pdg_id"])] += 1

    rows = []
    for (parent_pdg, child_pdg), count in counts.most_common():
        rows.append({
            "parent_pdg_id": parent_pdg,
            "parent": pdg_label(parent_pdg),
            "child_pdg_id": child_pdg,
            "child": pdg_label(child_pdg),
            "occurrences": count,
        })
    return rows

display(maybe_frame(scan_decay_signatures(tree, max_events=500)))


In [ ]:
def match_source_breakdown(tree, max_events=None):
    source_counter = Counter()
    chi2_values = []
    n_entries = tree.GetEntries() if max_events is None else min(tree.GetEntries(), max_events)

    for entry in range(n_entries):
        snap = event_snapshot(tree, entry)
        for row in snap["mu_rows"]:
            source_counter[row["match_source"]] += 1
            if row["match_source"] == "chi2Fallback" and row["match_chi2"] is not None:
                chi2_values.append(row["match_chi2"])

    display(maybe_frame([{"source": key, "count": value} for key, value in sorted(source_counter.items())]))
    if chi2_values:
        print({
            "fallback_matches": len(chi2_values),
            "chi2_min": min(chi2_values),
            "chi2_mean": sum(chi2_values) / len(chi2_values),
            "chi2_max": max(chi2_values),
        })

match_source_breakdown(tree)


## Suggested Follow-ups

- Change `ENTRY` to inspect a specific signal event.
- Use `walk_mother_chain()` on any `gen_idx` to inspect radiative self-chains explicitly.
- If reco kaon to GEN kaon matching is added later, mirror the muon-side tables for `phi` daughters.
